# SoundStaff 고품질 변환 (반주 제거 → MIDI)

음원(mp3, wav 등)에서 **반주를 제거해 보컬만 남기고**, 그 소리를 **MIDI 파일**로 바꿉니다.
만들어진 MIDI 파일을 SoundStaff 화면으로 가져가면 악보가 만들어집니다.

### 사용 방법 (코드를 몰라도 됩니다)
1. 위쪽 메뉴 **[런타임] → [런타임 유형 변경]** 에서 **T4 GPU**(또는 GPU)를 선택하고 저장합니다. (더 빨라집니다)
2. 아래 **1번 칸부터 순서대로**, 각 칸 왼쪽의 ▶ 버튼을 누릅니다. 앞 칸이 끝나야 다음 칸을 누를 수 있습니다.
3. **3번 칸**에서 [파일 선택]이 나오면 내 음원 파일을 고릅니다.
4. 맨 마지막 칸이 끝나면 **MIDI 파일이 자동으로 내려받아집니다.**
5. SoundStaff 화면으로 돌아와 받은 MIDI 파일을 놓으면 악보가 만들어집니다.

### 알아 두세요
- 내 음원은 **내 구글 계정의 Colab에만** 올라가며, SoundStaff 쪽으로는 전송되지 않습니다. 작업이 끝나면 Colab 연결을 끊거나 [런타임] → [세션 관리]에서 종료하면 올린 파일이 지워집니다.
- 무료 Colab에는 사용 시간 제한이 있어서, 이용이 몰리면 GPU를 못 쓰거나 잠시 막힐 수 있습니다. 그럴 땐 시간을 두고 다시 시도하세요.
- 사용 권한이 있는 음원만 올려 주세요.
- 자동 분석 결과는 정확하지 않을 수 있습니다. 악보 편집 프로그램(Finale 등)에서 고쳐 주세요.
- 사용 도구: Demucs(반주 분리, MIT 라이선스), Basic Pitch(음표 찾기, Apache-2.0 라이선스)

## 1. 도구 설치
처음 한 번 1~3분 걸립니다. 빨간 오류가 나오면 맨 아래 **[문제가 생겼을 때]** 를 보세요.

In [ ]:
# 1. 도구 설치 (Demucs: 반주 분리, Basic Pitch: 소리를 MIDI로)
!pip install -q -U demucs
!pip install -q -U basic-pitch
print("설치가 끝났습니다. 다음 칸으로 넘어가세요.")

## 2. 설정
기본값 그대로 두면 됩니다. 필요할 때만 바꾸세요.

- `반주제거`: `True`면 보컬만 남기고 변환합니다(권장). 이미 보컬만 있는 파일이면 `False`로 바꾸세요.

In [ ]:
# 2. 설정 (바꾸지 않아도 됩니다)
REMOVE_ACCOMPANIMENT = True  #@param {type:"boolean"}

## 3. 음원 파일 올리기
아래 칸을 실행하면 [파일 선택] 버튼이 나옵니다. mp3, wav, m4a, flac 등 음원 파일 **하나**를 고르세요.

In [ ]:
# 3. 음원 파일 올리기
from google.colab import files
import os, re

os.makedirs("/content/input", exist_ok=True)
uploaded = files.upload()
if not uploaded:
    raise SystemExit("파일이 선택되지 않았습니다. 이 칸을 다시 실행해 파일을 골라 주세요.")

orig_name = list(uploaded.keys())[0]
base_name, ext = os.path.splitext(orig_name)
# 파일 이름의 특수문자 때문에 도구가 실패하지 않도록 안전한 임시 이름을 쓴다 (내려받을 때는 원래 이름을 쓴다)
tmp_path = "/content/input/track" + ext.lower()
os.replace(orig_name, tmp_path)
print("올린 파일:", orig_name)

## 4. 반주 제거 (보컬만 남기기)
GPU에서 1~3분 걸립니다(곡 길이에 따라 다릅니다). 진행 막대가 나옵니다.

In [ ]:
# 4. 반주 제거 (Demucs). 보컬만 남깁니다.
import glob, torch

if REMOVE_ACCOMPANIMENT:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("GPU를 쓰지 못해 CPU로 진행합니다. 오래 걸릴 수 있습니다. ([런타임] → [런타임 유형 변경]에서 GPU를 선택하면 빨라집니다)")
    # (wav 저장은 최신 torchaudio에서 실패하는 경우가 있어 mp3로 저장한다. 변환 품질에는 영향이 거의 없다)
    !python -m demucs --two-stems=vocals --mp3 --mp3-bitrate 320 -n htdemucs -d {device} -o /content/separated {tmp_path}
    vocal_files = glob.glob("/content/separated/htdemucs/track/vocals.*")
    if not vocal_files:
        raise SystemExit("반주 제거에 실패했습니다. 아래 [문제가 생겼을 때]를 확인해 주세요.")
    source_file = vocal_files[0]
    print("반주 제거 완료:", source_file)
else:
    source_file = tmp_path
    print("반주 제거를 건너뜁니다. 올린 파일을 그대로 변환합니다.")

## 5. MIDI로 변환
몇십 초 걸립니다.

In [ ]:
# 5. MIDI로 변환 (Basic Pitch)
from basic_pitch.inference import predict_and_save
from basic_pitch import ICASSP_2022_MODEL_PATH

os.makedirs("/content/midi", exist_ok=True)
predict_and_save(
    [source_file],
    "/content/midi",
    save_midi=True,
    sonify_midi=False,
    save_model_outputs=False,
    save_notes=False,
    model_or_model_path=ICASSP_2022_MODEL_PATH,
)
midi_files = glob.glob("/content/midi/*.mid")
if not midi_files:
    raise SystemExit("MIDI를 만들지 못했습니다. 소리가 너무 작거나 비어 있는지 확인해 주세요.")
print("MIDI 변환 완료")

## 6. MIDI 파일 내려받기
이 칸을 실행하면 MIDI 파일이 **자동으로 내려받아집니다**(브라우저의 다운로드 폴더).
내려받은 뒤 SoundStaff 화면으로 돌아가 파일을 놓으세요.

In [ ]:
# 6. MIDI 파일 내려받기 (원래 음원 이름.mid)
dl_name = re.sub(r'[\\/:*?"<>|]+', "_", base_name) + ".mid"
final_path = "/content/midi/" + dl_name
os.replace(midi_files[0], final_path)
files.download(final_path)
print("내려받기: ", dl_name)
print("이제 SoundStaff 화면으로 돌아가 이 파일을 놓으세요.")

## 문제가 생겼을 때

- **1번 칸에서 빨간 오류**: [런타임] → [런타임 다시 시작]을 누른 뒤 1번 칸부터 다시 실행하세요. 그래도 안 되면 오류 문구를 SoundStaff 작성자에게 알려 주세요.
- **4번 칸이 너무 오래 걸림**: GPU가 꺼져 있을 수 있습니다. [런타임] → [런타임 유형 변경]에서 GPU를 선택하고 [런타임 다시 시작] 후 1번부터 다시 하세요.
- **"GPU 사용량 한도" 안내**: 무료 사용량이 찼습니다. 몇 시간 뒤에 다시 시도하세요.
- **6번 칸에서 다운로드가 안 됨**: 브라우저가 팝업을 막았을 수 있습니다. 주소창 오른쪽의 차단 표시를 눌러 허용한 뒤 6번 칸을 다시 실행하세요. 또는 왼쪽 폴더 아이콘 → `midi` 폴더에서 파일을 직접 내려받을 수 있습니다.
- **MIDI 음이 엉망**: 반주가 센 곡은 정확도가 낮습니다. 2번 칸의 `반주제거`가 `True`인지 확인하고, 가능하면 보컬이 분리된 파일을 올려 보세요.